# grid — the enhanced algorithm (thesis Sec. 6.2.4), pure-numpy reference

Compute **ONE** operator classically, then propagate arbitrarily far by only re-applying it and
re-reading the last state each step. Nothing here computes the dynamics "for every $t$" on the
classical computer — that is the whole point.

Column-stacking convention everywhere: $\mathrm{vec}(\rho)[i+dj]=\rho[i,j]$.

**Two variants, and which method needs which:**

1. `populations_enhanced` — Markovian enhanced algorithm. Needs ONLY the one-step map $\mathcal L(\Delta t)$:
   build the Kraus operators $M_k(\Delta t)$ once, then
   $\rho_n=\sum_k M_k(\Delta t)\,\rho_{n-1}\,M_k(\Delta t)^\dagger$ (on hardware: one dilated circuit,
   re-initialised with the measured state — cf. the thesis example `figure6.17.py`). Exact **iff** the
   map is a semigroup, $\mathcal L(n\Delta t)=\mathcal L(\Delta t)^n$ — true for **Lindblad** (GKSL is
   divisible). For a non-Markovian map it is an approximation: a system-only channel applied repeatedly
   can only ever produce a semigroup, so any memory beyond one step is necessarily dropped.

2. `populations_enhanced_memory` — memory-enhanced algorithm, keeps the non-Markovian memory. Needed
   for the **path integral** (and, via its short-time maps, for **HEOM**). Since a channel on the
   system alone cannot carry memory, the memory is put **on the register**:
   $$X_n=(\mathrm{vec}\,\rho_n,\ \mathrm{vec}\,\rho_{n-1},\ \dots,\ \mathrm{vec}\,\rho_{n-K+1}),$$
   propagated by **one fixed operator** $E$ built once from the transfer tensors $T_1..T_K$
   (Cerrillo & Cao, PRL **112**, 110401 (2014)),
   $$T_n=\mathcal L_n-\sum_{m=1}^{n-1}T_m\mathcal L_{n-m},$$
   learned from the SHORT-time maps $\mathcal L(\Delta t)..\mathcal L(K\Delta t)$ only — the classical
   cost covers just the memory time ($K\Delta t$), not the full propagation:
   $$X_{n+1}=EX_n,\qquad \rho_{n+1}=\text{first block of }X_{n+1}.$$
   The lower block rows of $E$ just shift the history window.

   On hardware, $E/s$ ($s\ge\|E\|_2$) becomes **one unitary** via the Sz.-Nagy dilation
   (`sz_nagy_dilation`) — the same trick as the thesis circuit (`figure6.17.py` `dilate`; cf.
   Head-Marsden *et al.*, Phys. Rev. Research **3**, 013182 (2021)) — and the loop is unchanged: apply
   the SAME dilated operator, read out, renormalise, feed the state back in.

**HEOM uses variant 2 as well.** Its one classical object is the extended propagator
$P=e^{\hat{\mathcal L}_{\rm HEOM}\Delta t}$ (one matrix exponential), but $P$ acts on the ADO hierarchy,
not on a density matrix, so it is not what the grid re-applies: $P$ is used once to produce the
short-time reduced maps $\mathcal L(\Delta t)..\mathcal L(K\Delta t)$ (`heom_map.heom_shorttime_maps`),
and those feed exactly the same transfer tensors $\to$ companion operator $\to$ dilated unitary as the
path integral.

In both variants the propagation itself runs on the grid: variant 1 by the Kraus operator sum, variant
2 by `propagate_dilated` (one unitary, applied over and over, reading the state out and feeding it back
in). Nothing is recomputed per time step. **This notebook is the pure-numpy reference** for what
`circuit.ipynb` runs as real Qiskit circuits.

In [ ]:
import numpy as np
from scipy.linalg import sqrtm


def vec(rho):
    return np.asarray(rho).flatten(order="F")


def unvec(v, d):
    return np.asarray(v).reshape(d, d, order="F")

# Choi / Kraus

### `choi_from_map`
$$C(t) := \sum_{i,j=1}^N (E_{i,j} \otimes I) \mathscr{L}(t)(I \otimes E_{i,j}) = \sum_{i,j=1}^N (E_{i,j} \otimes I) \mathscr{L}(t)(I \otimes E_{i,j}) = \sum_{i,j=1}^N \vert{}i\rangle\langle j\vert{} \otimes \mathcal{E}(\vert{}i\rangle\langle j\vert{}) = \sum_{i=0}^{d-1} \sum_{j=0}^{d-1} E_{ij} \otimes \text{unvec}\big(L \, \text{vec}(E_{ij})\big), \quad E_{i,j} = \vert{}i\rangle\langle j\vert{}$$

### `kraus_from_choi`
Zuerst wird die Choi-Matrix symmetrisiert, um numerische Fehler zu beheben und Hermitizität zu garantieren:$C_h = \frac{1}{2}\left(C + C^\dagger\right)$. Dann wird die Spektralzerlegung (Eigenwertzerlegung) berechnet:$C_h \vec{v}_k = \lambda_k \vec{v}_k$. Für alle Eigenwerte $\lambda_k > \text{tol}$ werden daraus die Kraus-Operatoren $M_k$ gebildet:$$M_k = \sqrt{\lambda_k} \cdot \text{unvec}(\vec{v}_k)$$

### `enhanced_kraus`
$M_k(dt) = \text{kraus\_from\_choi} \Big( \text{choi\_from\_map} \big( L(dt) \big) \Big)$

In [ ]:
def choi_from_map(L, d):
    """Choi matrix  C = sum_ij E_ij (x) L(E_ij)  (Seneviratne eq. 13)."""
    C = np.zeros((d * d, d * d), dtype=complex)
    for i in range(d):
        for j in range(d):
            Eij = np.zeros((d, d), dtype=complex)
            Eij[i, j] = 1.0
            C += np.kron(Eij, unvec(L @ vec(Eij), d))
    return C


def kraus_from_choi(C, d, tol=1e-10):
    """Kraus operators from the Choi eigendecomposition (eqs. 14-15).
    Returns (kraus_list, min_eigenvalue)."""
    Ch = 0.5 * (C + C.conj().T)
    evals, evecs = np.linalg.eigh(Ch)
    kraus = [np.sqrt(evals[k]) * unvec(evecs[:, k], d)
             for k in range(len(evals)) if evals[k] > tol]
    return kraus, float(evals.min())


def enhanced_kraus(L_dt, d, tol=1e-10):
    """The ONE-STEP Kraus operators M_k(dt) of the one-step map L(dt).
    This is the entire classical computation of variant 1: it is done once
    and never repeated. Returns (kraus_list, min_choi_eigenvalue)."""
    return kraus_from_choi(choi_from_map(L_dt, d), d, tol)

# Map files

Only the path integral needs one: its transfer tensors are learned from the short-time maps
$\mathcal L(0)..\mathcal L(K\Delta t)$, produced by `pathintegral_map.ipynb`. The following cells save the calculated data into a file or read it in again.

In [ ]:
def save_maps(path, t_fs, maps, H, label):
    H = np.asarray(H)
    np.savez(path, t_fs=np.asarray(t_fs, float),
             maps=np.asarray(maps, complex), H=H, d=H.shape[0],
             label=str(label))


def load_maps(path):
    """Returns dict with t_fs, maps, H, d, label."""
    z = np.load(path, allow_pickle=True)
    return dict(t_fs=z["t_fs"], maps=z["maps"], H=z["H"],
                d=int(z["d"]), label=str(z["label"]))

# Memory-enhanced algorithm (non-Markovian): transfer tensors + one fixed companion operator $E$

### Der `transfer_tensors`

Gegeben sei ein Anfangszustand $\rho_0$. Die Zeitentwicklung zu einem beliebigen Schritt $n$ wird durch eine Familie von dynamischen Karten (Superoperatoren) $\mathcal{E}_n$ beschrieben (das ist bei uns `maps`), sodass:$$\rho_n = \mathcal{E}_n \rho_0$$

Bei `maps` für HEOM sieht man ganz gut dass man $\rho_S(n\cdot \Delta t)$ bekommt wenn man den $n$-ten Eintrag von `maps` nimmt und ihn mit einem initial state multipliziert. Wenn das System Markovsch (gedächtnislos) wäre, bräuchte man nur den Zustand vom vorherigen Schritt, um den nächsten zu berechnen: $\rho_n = \mathcal{E}_1 \rho_{n-1}$.Da wir uns im nicht-Markovschen Regime befinden, hängt der aktuelle Zustand von der gesamten Vergangenheit ab. Wir suchen also eine diskrete Faltungsgleichung der Form:$$\rho_n = \sum_{m=1}^n T_m \rho_{n-m}$$Hier ist $T_m$ der Transfer-Tensor. Er quantifiziert exakt: "Wie stark beeinflusst der Zustand, der vor $m$ Zeitschritten herrschte, unseren aktuellen Zustand?" Wir können sagen dass $\rho_n$ eine Linearkombination von alten Zuständen ist, sowohl die unitäre Entwicklung $\rho_S(t) = \text{Tr}_B [ U(t) (\rho_S(0) \otimes \rho_B(0)) U^\dagger(t) ]$ als auch die Spur linear sind: Wenn du den Anfangszustand des Systems verdoppelst, verdoppelt sich auch $\rho_{S}(t)$. Man kann das elegant mit dem Nakajima-Zwanzig-Formalismus zeigen.

Um die Tensoren $T_m$ zu berechnen, setzen wir die Definition der dynamischen Karte ($\rho_k = \mathcal{E}_k \rho_0$) in unsere Faltungsgleichung ein:$$\mathcal{E}_n \rho_0 = \sum_{m=1}^n T_m \mathcal{E}_{n-m} \rho_0$$Da diese Gleichung für jeden beliebigen Anfangszustand $\rho_0$ gelten muss, können wir $\rho_0$ weglassen und erhalten eine Operatorgleichung:$$\mathcal{E}_n = \sum_{m=1}^n T_m \mathcal{E}_{n-m}$$Um nun den Tensor $T_n$ für den aktuellen Schritt zu isolieren, spalten wir den letzten Term (für $m=n$) von der Summe ab. Wir wissen, dass $\mathcal{E}_0 = \mathbb{I}$ (der nullte Schritt verändert das System nicht). Also ist der Term für $m=n$ einfach $T_n \mathcal{E}_0 = T_n$. Umgestellt nach $T_n$ ergibt das:$$T_n = \mathcal{E}_n - \sum_{m=1}^{n-1} T_m \mathcal{E}_{n-m}$$Hiermit wird aus der exakten Gesamtdynamik iterativ der reine Kurzzeit- und Gedächtnisanteil herausgefiltert. In der Praxis schneidet man diese Reihe nach $K$ Schritten ab, da das Bad irgendwann "vergisst" ($T_{m > K} \approx 0$).

Da $\mathcal{E}_n \equiv \mathcal{E}(n\Delta t)$ und somit $\mathcal{E}_0=1$ (Propagieren um 0 Zeitschritte macht nichts) folgt $\mathcal{E}_1 = \sum_{m=1}^1 T_m \mathcal{E}_{1-m} = T_1 \mathcal{E}_0 = T_1$. Daraus folgt wiederum:
$$T_2 = \mathcal{E}_2 - \sum_{m=1}^{2-1} T_m \mathcal{E}_{2-m} = \mathcal{E}_2 - T_1 \mathcal{E}_{1} = \mathcal{E}_2 - \mathcal{E}_{1} \mathcal{E}_{1}$$
$$T_3 = \mathcal{E}_3 - \sum_{m=1}^{2} T_m \mathcal{E}_{3-m} = \mathcal{E}_3 - \left( T_1 \mathcal{E}_2 + T_2 \mathcal{E}_1 \right) = \mathcal{E}_3 - \mathcal{E}_1 \mathcal{E}_2 - \left( \mathcal{E}_2 - \mathcal{E}_1 \mathcal{E}_1 \right) \mathcal{E}_1 = \mathcal{E}_3 - \mathcal{E}_1 \mathcal{E}_2 - \mathcal{E}_2 \mathcal{E}_1 + \mathcal{E}_1 \mathcal{E}_1 \mathcal{E}_1$$

$$\begin{aligned} T_4 &= \mathcal{E}_4 - \sum_{m=1}^{3} T_m \mathcal{E}_{4-m} = \mathcal{E}_4 - \left( T_1 \mathcal{E}_3 + T_2 \mathcal{E}_2 + T_3 \mathcal{E}_1 \right) \\ &= \mathcal{E}_4 - \mathcal{E}_1 \mathcal{E}_3 - \left( \mathcal{E}_2 - \mathcal{E}_1 \mathcal{E}_1 \right) \mathcal{E}_2 - \left( \mathcal{E}_3 - \mathcal{E}_1 \mathcal{E}_2 - \mathcal{E}_2 \mathcal{E}_1 + \mathcal{E}_1 \mathcal{E}_1 \mathcal{E}_1 \right) \mathcal{E}_1 \\ &= \mathcal{E}_4 - \mathcal{E}_1 \mathcal{E}_3 - \mathcal{E}_2 \mathcal{E}_2 + \mathcal{E}_1 \mathcal{E}_1 \mathcal{E}_2 - \mathcal{E}_3 \mathcal{E}_1 + \mathcal{E}_1 \mathcal{E}_2 \mathcal{E}_1 + \mathcal{E}_2 \mathcal{E}_1 \mathcal{E}_1 - \mathcal{E}_1 \mathcal{E}_1 \mathcal{E}_1 \mathcal{E}_1 \end{aligned}$$

### Der `companion_propagator`
Die Transfer-Tensor-Gleichung, die wir gerade hergeleitet haben, benötigt aber exakt diese Historie:$$\rho_n = T_1 \rho_{n-1} + T_2 \rho_{n-2} + \dots + T_K \rho_{n-K}$$Um das auf einem Quantencomputer (oder Simulator) laufen zu lassen, nutzt die Funktion companion_propagator einen eleganten Trick der linearen Algebra: Sie wandelt die Rekursionsgleichung $K$-ter Ordnung in eine Gleichung 1. Ordnung um, indem sie den Zustandsraum künstlich vergrößert.

Anstatt nur die aktuelle Dichtematrix $\rho$ zu betrachten, definieren wir einen großen Blockvektor $X$, der ein "gleitendes Fenster" (sliding window) der letzten $K$ Zustände enthält. Wenn wir uns im Schritt $n-1$ befinden, sieht unser Input-Register so aus:$$X_{n-1} = \begin{pmatrix} \rho_{n-1} \\ \rho_{n-2} \\ \rho_{n-3} \\ \vdots \\ \rho_{n-K} \end{pmatrix}$$Unser Ziel ist es, einen einzigen Operator $E$ zu finden, der diesen Vektor einen Zeitschritt in die Zukunft propagiert: $X_n = E X_{n-1}$.

Der Operator $E$ muss zwei Aufgaben gleichzeitig erfüllen:
- Den neuen Zustand $\rho_n$ berechnen und in die oberste Position schreiben.
- Alle alten Zustände um eine Position nach unten verschieben (wobei der älteste Zustand $\rho_{n-K}$ aus dem Register "herausfällt").

Dies geschieht durch eine sogenannte Block-Begleitmatrix (Companion Matrix). Sie hat folgende Struktur:$$E = \begin{pmatrix}  T_1 & T_2 & T_3 & \dots & T_K \\  \mathbb{I} & 0 & 0 & \dots & 0 \\  0 & \mathbb{I} & 0 & \dots & 0 \\  \vdots & \vdots & \ddots & \ddots & \vdots \\  0 & 0 & \dots & \mathbb{I} & 0  \end{pmatrix}$$

Wenn du nun das Matrix-Vektor-Produkt $E X_{n-1}$ im Kopf ausführst, siehst du, was passiert:
- Die erste Zeile multipliziert sich mit dem Vektor zu: $T_1 \rho_{n-1} + T_2 \rho_{n-2} + \dots + T_K \rho_{n-K}$. Das ist genau unser gesuchtes $\rho_n$.
- Die zweite Zeile besteht nur aus einer Identitätsmatrix $\mathbb{I}$ an der ersten Position. Sie greift sich also einfach $\rho_{n-1}$ und schiebt es in die zweite Zeile des neuen Vektors.
- Die dritte Zeile greift sich $\rho_{n-2}$ und so weiter.

Die Funktion `companion_propagator` baut exakt diese Matrix $E$ in zwei simplen For-Schleifen auf. D ist dabei die Dimension des ursprünglichen Hilbertraums (bzw. Liouville-Raums), also die Größe eines einzelnen $\rho$. Es wird über alle berechneten Transfer-Tensoren iteriert. Sie werden nebeneinander in die allerersten D Zeilen (die erste Blockzeile E[:D, ...]) der leeren Matrix eingefügt.

In [ ]:
def transfer_tensors(maps, K):
    """Transfer tensors T_1..T_K from the short-time maps L_1..L_K only."""
    T = []
    for n in range(1, K + 1):
        Tn = np.array(maps[n], dtype=complex, copy=True)
        for m in range(1, n):
            Tn -= T[m - 1] @ maps[n - m]
        T.append(Tn)
    return T


def companion_propagator(T):
    """The ONE fixed one-step operator E on the enlarged (K*D-dim) register:
    first block row = (T_1 ... T_K), lower rows shift the history window."""
    K, D = len(T), T[0].shape[0]
    E = np.zeros((K * D, K * D), dtype=complex)
    for m, Tm in enumerate(T):
        E[:D, m * D:(m + 1) * D] = Tm
    for r in range(1, K):
        E[r * D:(r + 1) * D, (r - 1) * D:r * D] = np.eye(D)
    return E

---
## What is deliberately *not* here

`sz_nagy_dilation`, `propagate_dilated`, `populations_enhanced` and
`populations_enhanced_memory` used to live in this notebook. They are a **pure-numpy
re-implementation of the propagation** — no circuit, no simulator — kept only to check the
Qiskit result against. They now live in **`validation.ipynb`** so that this notebook contains
exactly the pieces the real quantum grid (`circuit.ipynb`) is built from:

| here (used by the grid) | in `validation.ipynb` (reference only) |
|---|---|
| `vec`, `unvec` | `sz_nagy_dilation` |
| `choi_from_map`, `kraus_from_choi`, `enhanced_kraus` | `propagate_dilated` |
| `transfer_tensors`, `companion_propagator` | `populations_enhanced` |
| `save_maps`, `load_maps` | `populations_enhanced_memory` |

`circuit.ipynb` does its own Sz.-Nagy dilation inside `DilatedChannel` (it has to pad the
register to a power of two first), so the grid never imports anything from `validation.ipynb`.